<a href="https://colab.research.google.com/github/saurabh202/Capstone_Project_code/blob/main/Capstone_code_file.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Mount Google Drive to use Dataset

Dataset link : https://drive.google.com/drive/folders/1YC4D7eTf1_phvQvq9bbTL5YS_GcxL65C?usp=share_link


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os

os.makedirs('/content/dataset', exist_ok=True)
!unzip -q "/content/drive/MyDrive/Capstone_Dataset/AAAI_dataset.zip" -d /content/dataset

# Verify
print('Dataset files:', os.listdir('/content/dataset/AAAI_dataset'))
print('Train images:', len(os.listdir('/content/dataset/AAAI_dataset/Images/gossip_train')))
print('Test images: ', len(os.listdir('/content/dataset/AAAI_dataset/Images/gossip_test')))

Dataset files: ['gossip_train.csv', 'Images', 'gossip_test.csv', 'politi_train.csv', 'gossip_test.xlsx', '.DS_Store', 'gossip_train.xlsx', 'politi_test.csv']
Train images: 9988
Test images:  2828


## Clone the fork from MIMoE-FND



In [5]:
%cd /content
!git clone https://github.com/saurabh202/Capstone_Project_code.git MIMoE-FND
%cd /content/MIMoE-FND
!ls

/content
Cloning into 'MIMoE-FND'...
remote: Enumerating objects: 72, done.
remote: Counting objects: 100% (72/72), done.
remote: Compressing objects: 100% (70/70), done.
remote: Total 72 (delta 15), reused 40 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (72/72), 2.34 MiB | 23.28 MiB/s, done.
Resolving deltas: 100% (15/15), done.
/content/MIMoE-FND
Capstone_code_file.ipynb  models_mae.py  requirements.txt  util
data			  preprocessing  train_vimoe.py    visualizations
models			  README.md	 train_vimoe.sh


## Install Dependencies

In [ ]:
!pip install -q pytorch-warmup paramiko positional-encodings ftfy timm einops
!pip install -q git+https://github.com/openai/CLIP.git

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.9/208.9 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.0/161.0 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 64.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


## MAE Pretrained Download



Source: https://github.com/facebookresearch/mae

In [ ]:
%cd /content/MIMoE-FND
!wget -q --show-progress https://dl.fbaipublicfiles.com/mae/pretrain/mae_pretrain_vit_base.pth
!ls -lh mae_pretrain_vit_base.pth

/content/MIMoE-FND
mae_pretrain_vit_ba 100%[===================>] 327.35M   216MB/s    in 1.5s    
-rw-r--r-- 1 root root 328M Dec 29  2021 mae_pretrain_vit_base.pth


## Checkpoint for recording accuracies



In [ ]:
!mkdir -p /content/MIMoE-FND/checkpoints/gossip
print('Checkpoint directory ready.')

Checkpoint directory ready.



##  Training batch_size=4 , epoch=1


In [ ]:
%cd /content/MIMoE-FND

!python train_vimoe.py \
    -train_dataset gossip \
    -test_dataset  gossip \
    -batch_size    4      \
    -epochs        1      \
    -val           0      \
    -duplicate_fake_times 0 \
    -device        cuda:0 \
    -int_lr        1e-6   \
    -int_beta      0.1    \
    -agr_threshold 0.3    \
    -sem_threshold 0.3    \

/content/MIMoE-FND
tokenizer_config.json: 100% 49.0/49.0 [00:00<00:00, 225kB/s]
vocab.txt: 100% 110k/110k [00:00<00:00, 1.72MB/s]
tokenizer.json: 100% 269k/269k [00:00<00:00, 4.08MB/s]
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 255kB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 2.64MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 3.18MB/s]
preprocessor_config.json: 100% 316/316 [00:00<00:00, 2.16MB/s]
config.json: 100% 4.10k/4.10k [00:00<00:00, 14.8MB/s]
tokenizer_config.json: 100% 905/905 [00:00<00:00, 6.10MB/s]
vocab.json: 100% 961k/961k [00:00<00:00, 48.4MB/s]
merges.txt: 100% 525k/525k [00:00<00:00, 97.9MB/s]
tokenizer.json: 100% 2.22M/2.22M [00:00<00:00, 136MB/s]
special_tokens_map.json: 100% 389/389 [00:00<00:00, 1.98MB/s]
Namespace(output_file='./output', train_dataset='gossip', test_dataset='gossip', checkpoint='', device='cuda:0', finetune=0, val=0, duplicate_fake_times=0, batch_size=4, epochs=1, hidden_dim=512, embed_dim=32, vocab_size=25, text_only=False, get_MLP_sc

##KG implementation



In [1]:
!pip install -q transformers

## Load data and to detect text column

In [7]:
import pandas as pd

train_df = pd.read_excel('/content/dataset/AAAI_dataset/gossip_train.xlsx')
test_df  = pd.read_excel('/content/dataset/AAAI_dataset/gossip_test.xlsx')

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Columns:", train_df.columns.tolist())
train_df.head(2)

Train shape: (9759, 5)
Test shape: (2770, 5)
Columns: ['Unnamed: 0', 'content', 'image', 'label', 'type']


,Unnamed: 0,content,image,label,type
0,0,One Love Manchester: Katy Perry wears photos o...,DyweiEOWJPtOqslc31CyVnTCyvAm307J.jpg,1,multi
1,1,TLC cuts all ties with Derick Dillard after mo...,NSq6JnA5ziOJbAQCzVlfti3OymZvInGK.jpg,1,multi


In [8]:
candidates = ['content', 'text', 'caption', 'title', 'news', 'article']
text_col = next((c for c in candidates if c in train_df.columns), None)

if text_col is None:
    raise ValueError(f"Couldn't auto-detect text column. Available columns: {train_df.columns.tolist()}")

print(f"Using text column: '{text_col}'")

Using text column: 'content'


##  Load the NER pipeline

In [11]:
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

device = 0 if torch.cuda.is_available() else -1

tokenizer = AutoTokenizer.from_pretrained("dslim/bert-base-NER")
model = AutoModelForTokenClassification.from_pretrained("dslim/bert-base-NER")

ner_pipeline = pipeline(
    "ner",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="max",   # changed from "simple" — fixes subword fragment merging
    device=device
)

print("NER pipeline loaded. Using device:", "GPU" if device == 0 else "CPU")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


NER pipeline loaded. Using device: GPU


## Entity Extraction on train and test

In [12]:
from tqdm import tqdm

def extract_entities(text):
    if not isinstance(text, str) or not text.strip():
        return []
    try:
        results = ner_pipeline(text[:512])
        return [(r['word'], r['entity_group'], float(r['score'])) for r in results]
    except Exception as e:
        return []

tqdm.pandas()

train_df['entities'] = train_df['content'].progress_apply(extract_entities)
test_df['entities']  = test_df['content'].progress_apply(extract_entities)

print(train_df[['content', 'entities']].head(3))

100%|██████████| 2770/2770 [00:45<00:00, 60.42it/s]

                                             content  \
0  One Love Manchester: Katy Perry wears photos o...   
1  TLC cuts all ties with Derick Dillard after mo...   
2  Cher Steals the Show at the 2017 Billboard Mus...   

                                            entities  
0  [(Manchester, MISC, 0.5110037326812744), (Katy...  
1  [(TLC, ORG, 0.9967092275619507), (Derick Dilla...  
2  [(Cher, PER, 0.9955215454101562), (Billboard M...  


In [13]:
import os
os.makedirs('/content/MIMoE-FND/kg_data', exist_ok=True)

train_df[['entities']].to_pickle('/content/MIMoE-FND/kg_data/train_entities.pkl')
test_df[['entities']].to_pickle('/content/MIMoE-FND/kg_data/test_entities.pkl')

print("Saved entity extractions.")

Saved entity extractions.
